# 03 — Baseline classica HOG + SVM
Binaria: casco vs no_helmet. Mostra feature progettate a mano + classificatore classico.

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from src.config import set_reproducibility, select_device, SH17_CLASSES
set_reproducibility(42); print('device:', select_device('auto'))

In [ ]:
import yaml
from src.data import extract_crops_for_binary
from src.features import HOGConfig, extract_hog_batch
from src.classical_model import SVMConfig, train_svm, evaluate_classifier, save_model
from src.config import CLASS_TO_ID
from sklearn.model_selection import train_test_split

cfg = yaml.safe_load((root / 'configs/baseline_svm.yaml').read_text())
X_img, y = extract_crops_for_binary(root / 'data/processed/sh17/images/train',
    CLASS_TO_ID[cfg['positive_class']], CLASS_TO_ID[cfg['negative_class']],
    crop_size=tuple(cfg['crop_size']), helmet_iou_exclude=cfg['helmet_iou_exclude'],
    max_per_class=cfg['max_per_class'])
print('crops:', len(y), 'positives:', int(y.sum()))

In [ ]:
X = extract_hog_batch(X_img, HOGConfig())
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=cfg['val_fraction'], stratify=y, random_state=42)
model = train_svm(Xtr, ytr, SVMConfig(**{k: cfg['svm'][k] for k in ('kernel','C','gamma','class_weight')}))
metrics = evaluate_classifier(model, Xval, yval); metrics

In [ ]:
from src.visualization import plot_confusion_matrix
plot_confusion_matrix(metrics['confusion_matrix'], ['no_helmet','helmet'], root / 'reports/figures/baseline_confusion_matrix.png')
save_model(model, root / 'models/hog_svm_baseline.pkl')